# Regime-Stratified Sensitivity Analysis — Level 5L6 (isothermal)

Sensitivity of the L5L6 model (defective oxide L3 + defective metal L4 + surface kinetics L6)
**conditioned on the rate-limiting regime**, run **isothermally** at several temperatures.

## Regimes — surface / oxide / metal

`regime = argmax` of the flux-weighted resistance fractions
(`frac_surface`, `frac_oxide`, `frac_metal`), all three of which are **series** resistances.

> **Note — the parallel path is computed but not part of the label.**
> `calculate_full_model_flux_L346_v2` does model the defect bypass (pinhole / crack /
> grain-boundary, each with its own permeance), and `flux_intact` / `flux_defect` are recorded.
> But unlike the L5 study — where `defect` is a peer regime — the L5L6 label is argmax over the
> three series terms only. A run whose flux is mostly through pinholes is still labelled by
> whichever series resistance dominates the *intact* path. So an L5L6 `metal` cluster is NOT
> the same population as an L5 `metal` cluster; do not compare the two directly.

## Why isothermal

With `temperature` varying it monopolises the variance of log10(flux). A dummy-parameter test
on varying-T L5L6 clusters put temperature at **3.4-4.3x** the noise floor and, in the metal
cluster, **nothing else above 1.0x** — the runner-up (`crack_thickness_factor`) scored 0.84x,
below a random input the model never saw. L5L6 is worse than L5 here because it spreads 36
parameters over the residual variance instead of 28.

Sampling harder does not help: the floor is flat in n (0.085 at n=400, 0.092 at n=1500),
because delta's bias for an irrelevant input does not shrink with sample size. Pinning the
temperature is what lets the remaining 35 parameters compete.

Every result table carries a **`floor`** column (max delta over the dummy inputs) and
**`delta_over_floor`**. **A value at or below 1.0 is not resolved.**

> **Caveat for `theta`:** temperature does *not* dominate that metric (1.54x oxide, 0.88x
> metal), because theta is bounded [0,1] and analysed linearly. `K_eq_ref` and `H_eq` — the
> actual surface-kinetics parameters — do surface there. So theta was the one L5L6 output the
> varying-T run handled reasonably.

## Targeting

Presets are what create the study: an untargeted sweep yields essentially no surface-limited
rows. Yields measured at fixed temperature (250-draw probes) — note the surface yield more than
doubles across the range, so draw counts are sized per temperature:

| T | surface | oxide | metal |
|---|---|---|---|
| 773 K | 23.2% | 78.0% | 77.2% |
| 1073 K | 47.6% | 83.6% | 85.2% |
| 1273 K | 56.8% | 82.0% | 86.4% |

Run with the **`mace_env`** kernel (SALib pinned to 1.5.0).

In [ ]:
import os, sys, importlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)

parent_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
# This notebook must run with its OWN folder as the working directory: that is how
# calculations.config.model_config finds this folder's study_config.py, and how
# RESULTS_DIR below resolves. Fails loudly rather than silently using another study.
from calculations.config import assert_study_is_local
print(assert_study_is_local())

# Figures from this notebook go in one folder beside it.
from calculations.figures import results_dir, save_panels
FIGS = results_dir("sensitivity_regime_L5L6")

In [2]:
import calculations.sensitivity as sens
importlib.reload(sens)
from calculations.config.model_config import (
    SUGGESTED_RANGES_LEVEL5L6, REGIME_PRESETS, DEFAULT_SWEEP_TEMPERATURES, ACTIVE_STUDY
)
from calculations.sensitivity import (
    REGIME_LABELS, REGIME_SA_METRICS, LOG_METRICS_DEFAULT, presets_without,
    level5L6_model_wrapper, run_targeted_regime_scans, load_regime_scans,
    plot_regime_exploration, givendata_sensitivity_by_regime, summarize_givendata,
    plot_givendata_results, regime_comparison_matrix, plot_regime_comparison_heatmap,
    plot_regime_geometry
)

# temperature is PINNED, not sampled -> it leaves the SA input set entirely
PRESETS = presets_without(REGIME_PRESETS, 'temperature')
PARAMS  = [p for p in SUGGESTED_RANGES_LEVEL5L6 if p != 'temperature']
print(f"study: {ACTIVE_STUDY}")
print(f"{len(PARAMS)} sampled parameters (temperature pinned); regimes: {REGIME_LABELS}")

study: incoloy802_cr2o3
35 sampled parameters (temperature pinned); regimes: ('surface', 'oxide', 'metal')


In [3]:
# === Run configuration ========================================================
SEED         = 42
TEMPERATURES = list(DEFAULT_SWEEP_TEMPERATURES)      # 773 / 1073 / 1273 K
METRICS      = REGIME_SA_METRICS                     # ['flux', 'permeability', 'theta']
TOP_K        = 8

# Draw counts are no longer precomputed from a cached yield table.
# Passing N_per_regime=None makes run_targeted_regime_scans probe each
# preset and size the run from the yield it actually observes, which
# stays correct if a range, a preset or the active study changes.
N_BY_T = {T: None for T in TEMPERATURES}

RESULTS_DIR     = "sa_results"
FORCE_RECOMPUTE = False
os.makedirs(RESULTS_DIR, exist_ok=True)


## Phase 1 — Targeted per-regime scans, one set per temperature

Cached to `sa_results/scans/T<K>/scan_<regime>.csv`. The cache is only reused when every file
holds exactly the draw count currently requested — existence alone would silently serve a
previous configuration.

In [4]:
def cache_is_valid(paths, n_expected):
    """Reusable only if every scan exists, and matches the count if one was asked for.

    With auto-sizing `n_expected` is None: the draw count is decided by the probe
    at run time, so existence is all that can be checked. Set FORCE_RECOMPUTE to
    re-probe and re-scan.
    """
    for r, p in paths.items():
        if not os.path.exists(p):
            return False, f"{os.path.basename(p)} missing"
        if n_expected is None:
            continue
        n_have = len(pd.read_csv(p, usecols=[0]))
        if n_have != n_expected[r]:
            return False, (f"{os.path.basename(p)} holds {n_have} draws but config "
                           f"asks for {n_expected[r]}")
    return True, "every scan matches the requested draw counts"


clusters_T, scans_T, yields_T = {}, {}, {}

for T in TEMPERATURES:
    scan_dir = os.path.join(RESULTS_DIR, "scans", f"T{int(T)}")
    os.makedirs(scan_dir, exist_ok=True)
    paths = {r: os.path.join(scan_dir, f"scan_{r}.csv") for r in REGIME_LABELS}
    reusable, why = cache_is_valid(paths, N_BY_T[T])

    print(f"\n{'='*70}\nT = {T:.0f} K\n{'='*70}")
    if (not FORCE_RECOMPUTE) and reusable:
        print(f"  loading cached scans ({why})")
        cl, sc, _ = load_regime_scans(scan_dir, regimes=REGIME_LABELS)
        yl = {r: len(cl[r]) / len(sc[r]) for r in cl}
    else:
        print("  recomputing —", "FORCE_RECOMPUTE is set" if FORCE_RECOMPUTE else why)
        cl, sc, _, yl = run_targeted_regime_scans(
            N_per_regime=N_BY_T[T], regimes=REGIME_LABELS,
            presets=PRESETS, wrapper=level5L6_model_wrapper,
            fixed_params={'temperature': T},
            seed=SEED, save_dir=scan_dir, verbose=False)
    clusters_T[T], scans_T[T], yields_T[T] = cl, sc, yl
    print("  clusters:", {r: len(cl[r]) for r in REGIME_LABELS})
    print("  yields:  ", {r: f"{yl[r]*100:.1f}%" for r in REGIME_LABELS})

master = pd.concat(
    [pd.concat(clusters_T[T].values()).assign(temperature_K=T) for T in TEMPERATURES],
    ignore_index=True)
master.to_csv(os.path.join(RESULTS_DIR, "master_clusters.csv"), index=False)
print(f"\nmaster_clusters.csv: {len(master)} rows")


T = 773 K
  recomputing — scan_surface.csv missing
  saved scan -> sa_results/scans/T773/scan_surface.csv
  saved scan -> sa_results/scans/T773/scan_oxide.csv
  saved scan -> sa_results/scans/T773/scan_metal.csv
  clusters: {'surface': 1452, 'oxide': 1713, 'metal': 1735}
  yields:   {'surface': '21.9%', 'oxide': '77.4%', 'metal': '77.6%'}

T = 1073 K
  recomputing — scan_surface.csv missing
  saved scan -> sa_results/scans/T1073/scan_surface.csv
  saved scan -> sa_results/scans/T1073/scan_oxide.csv
  saved scan -> sa_results/scans/T1073/scan_metal.csv
  clusters: {'surface': 1804, 'oxide': 1706, 'metal': 1667}
  yields:   {'surface': '51.0%', 'oxide': '82.7%', 'metal': '83.1%'}

T = 1273 K
  recomputing — scan_surface.csv missing
  saved scan -> sa_results/scans/T1273/scan_surface.csv
  saved scan -> sa_results/scans/T1273/scan_oxide.csv
  saved scan -> sa_results/scans/T1273/scan_metal.csv
  clusters: {'surface': 1659, 'oxide': 1729, 'metal': 1702}
  yields:   {'surface': '57.7%', 'o

In [5]:
# Median outputs by temperature and regime — the physical sanity check
print(master.groupby(['temperature_K', 'regime'])[METRICS].median().to_string())

                               flux  permeability     theta
temperature_K regime                                       
773.0         metal    1.924101e-07  2.178452e-13  0.976024
              oxide    3.115604e-10  4.072064e-17  0.975940
              surface  8.913780e-11  1.045139e-13  0.008420
1073.0        metal    7.574924e-05  7.871986e-11  0.984838
              oxide    5.839745e-07  6.851637e-14  0.987441
              surface  6.587325e-09  6.850185e-12  0.001776
1273.0        metal    6.664876e-04  6.865369e-10  0.983622
              oxide    1.174128e-05  1.569290e-12  0.988741
              surface  3.259992e-08  3.360373e-11  0.000462


In [ ]:
for T in TEMPERATURES:
    print(f"\n===== T = {T:.0f} K =====")
    _panels = plot_regime_exploration(clusters_T[T], output_metrics=tuple(METRICS),
                            scan_df=pd.concat(scans_T[T].values(), ignore_index=True),
                            cross_col='system_rate_limiting')
    save_panels(_panels, f"regime_exploration_{T:.0f}K", FIGS, module='calculations/sensitivity.py')

## Phase 2 — Route B: given-data PAWN + delta, per temperature

Zero extra model evaluations. Each `(regime, metric)` entry carries the dummy-parameter noise
`floor`; `delta_over_floor <= 1` means the parameter is **not resolved**.

In [7]:
gd_T, tidy_T = {}, {}
for T in TEMPERATURES:
    print(f"\n===== T = {T:.0f} K =====")
    gd_T[T] = givendata_sensitivity_by_regime(
        clusters_T[T], PARAMS, output_metrics=METRICS, methods=('pawn', 'delta'),
        seed=SEED, log_metrics=LOG_METRICS_DEFAULT, n_dummy=20, verbose=True)
    tidy_T[T] = summarize_givendata(gd_T[T], PARAMS).assign(temperature_K=T)

tidy = pd.concat(tidy_T.values(), ignore_index=True)
tidy.to_csv(os.path.join(RESULTS_DIR, "routeB_givendata.csv"), index=False)
print(f"\nrouteB_givendata.csv: {len(tidy)} rows")


===== T = 773 K =====
  [surface/flux] n= 1452  floor=0.096  top-δ: P_upstream (0.281)
  [surface/permeability] n= 1452  floor=0.091  top-δ: metal_thickness (0.146)
  [surface/theta] n= 1452  floor=0.090  top-δ: H_eq (0.200)
  [oxide/flux] n= 1713  floor=0.094  top-δ: H_sol_ox (0.252)
  [oxide/permeability] n= 1713  floor=0.087  top-δ: H_sol_ox (0.246)
  [oxide/theta] n= 1713  floor=0.119  top-δ: H_eq (0.193)
  [metal/flux] n= 1735  floor=0.095  top-δ: K_s_ref (0.163)
  [metal/permeability] n= 1735  floor=0.092  top-δ: K_s_ref (0.189)
  [metal/theta] n= 1735  floor=0.107  top-δ: H_eq (0.206)

===== T = 1073 K =====
  [surface/flux] n= 1804  floor=0.098  top-δ: P_upstream (0.249)
  [surface/permeability] n= 1804  floor=0.091  top-δ: f_pinhole (0.161)
  [surface/theta] n= 1804  floor=0.106  top-δ: H_sol_ox (0.219)
  [oxide/flux] n= 1706  floor=0.091  top-δ: H_sol_ox (0.499)
  [oxide/permeability] n= 1706  floor=0.093  top-δ: H_sol_ox (0.492)
  [oxide/theta] n= 1706  floor=0.103  top-δ: 

In [8]:
# delta/floor for EVERY parameter, no threshold. A hard filter creates an artificial
# cliff (0.99 hidden, 1.01 shown) and hides how wide the noise band is.
for T in TEMPERATURES:
    for reg in REGIME_LABELS:
        sub = (tidy[(tidy.metric == 'flux') & (tidy.temperature_K == T)
                    & (tidy.regime == reg)]
               .sort_values('delta_over_floor', ascending=False))
        floor = sub['floor'].iloc[0]
        n_res = int((sub.delta_over_floor > 1.0).sum())
        print(f"  T={T:.0f}K  {reg:8s} floor={floor:.3f}  ({n_res}/{len(sub)} above 1.0x)")
        for _, r_ in sub.head(6).iterrows():
            mark = '' if r_.delta_over_floor > 1.0 else '   <- at/below noise'
            print(f"        {r_.parameter:22s} delta={r_.delta:.3f}"
                  f"  ({r_.delta_over_floor:.2f}x){mark}")
    print()

  T=773K  surface  floor=0.096  (8/35 above 1.0x)
        P_upstream             delta=0.281  (2.91x)
        trap_vacancy_E_b       delta=0.126  (1.31x)
        metal_thickness        delta=0.121  (1.26x)
        D_ref                  delta=0.117  (1.21x)
        H_s                    delta=0.114  (1.19x)
        trap_carbide_E_b       delta=0.103  (1.07x)
  T=773K  oxide    floor=0.094  (3/35 above 1.0x)
        H_sol_ox               delta=0.252  (2.67x)
        oxide_thickness        delta=0.177  (1.88x)
        D_ox_ref               delta=0.112  (1.18x)
        K_ox_ref               delta=0.093  (0.99x)   <- at/below noise
        E_diss                 delta=0.093  (0.99x)   <- at/below noise
        D_ref                  delta=0.093  (0.99x)   <- at/below noise
  T=773K  metal    floor=0.095  (11/35 above 1.0x)
        K_s_ref                delta=0.163  (1.72x)
        trap_vacancy_N_T       delta=0.142  (1.51x)
        metal_thickness        delta=0.128  (1.35x)
        D

In [ ]:
for reg in REGIME_LABELS:
    _panels = plot_givendata_results(gd_T[TEMPERATURES[-1]], reg, 'flux', PARAMS, top_n=5, temperature=TEMPERATURES[-1])
    save_panels(_panels, f"givendata_{reg}_flux", FIGS, module='calculations/sensitivity.py')

## Phase 2 (cont.) — regime geometry (highest temperature)

Clusters projected on their two strongest drivers. Dashed boxes are the 10-90 percentile band;
overlap is the reminder that regime boundaries are curved, not rectangular.

In [ ]:
T_show = TEMPERATURES[-1]
def regime_topk(reg, k=TOP_K, metric='flux', T=None):
    """Top-k parameters for one regime at one temperature, ranked by Route-B delta."""
    T = T_show if T is None else T
    sub = tidy[(tidy.temperature_K == T) & (tidy.regime == reg) & (tidy.metric == metric)]
    return sub.nlargest(k, 'delta')['parameter'].tolist()

for reg in REGIME_LABELS:
    _panels = plot_regime_geometry(clusters_T[T_show], axes=regime_topk(reg)[:2],
                         log_x=True, log_y=True, drivers_of=reg)
    save_panels(_panels, f"regime_geometry_{reg}", FIGS, module='calculations/sensitivity.py')

## Phase 3 — Cross-regime comparison, per temperature

Columns are column-normalised so each regime spans a full 0-1 scale; the small corner number is
the raw index. Exported `compare_*.csv` files are always raw.

In [ ]:
for T in TEMPERATURES:
    print(f"\n===== T = {T:.0f} K — delta, log10(flux) =====")
    _panels = plot_regime_comparison_heatmap(gd_T[T], 'flux', PARAMS, index='delta', top_union=5, temperature=T)
    save_panels(_panels, f"comparison_heatmap_{T:.0f}K", FIGS, module='calculations/sensitivity.py')
    for idx in ('delta', 'pawn_median'):
        for met in METRICS:
            mat = regime_comparison_matrix(gd_T[T], met, PARAMS, index=idx, top_union=5)
            if not mat.empty:
                mat.to_csv(os.path.join(RESULTS_DIR, f"compare_{idx}_{met}_T{int(T)}.csv"))

In [12]:
# How does each parameter's margin over the noise floor move with temperature?
# Unfiltered: every parameter appears at every T, so a value near 1.0 reads as
# "at the noise floor" instead of vanishing into a NaN that looks like missing data.
piv = (tidy[tidy.metric == 'flux']
       .pivot_table(index=['regime', 'parameter'], columns='temperature_K',
                    values='delta_over_floor'))
piv.columns = [f"{c:.0f}K" for c in piv.columns]
piv['best'] = piv.max(axis=1)

print("delta_over_floor for log10(flux) — ALL parameters, no threshold")
print("  1.0 = exactly at the dummy-parameter noise floor; higher = resolved\n")
for reg in REGIME_LABELS:
    fl = (tidy[(tidy.metric == 'flux') & (tidy.regime == reg)]
          .groupby('temperature_K')['floor'].first())
    print(f"[{reg}]  floor: " + "  ".join(f"{T:.0f}K={v:.3f}" for T, v in fl.items()))
    print(piv.loc[reg].sort_values('best', ascending=False).round(2).head(12).to_string())
    print()

piv.to_csv(os.path.join(RESULTS_DIR, "delta_over_floor_by_T.csv"))
print("saved delta_over_floor_by_T.csv")

delta_over_floor for log10(flux) — ALL parameters, no threshold
  1.0 = exactly at the dummy-parameter noise floor; higher = resolved

[surface]  floor: 773K=0.096  1073K=0.098  1273K=0.101
                  773K  1073K  1273K  best
parameter                                 
P_upstream        2.91   2.53   1.78  2.91
f_pinhole         0.92   1.65   1.93  1.93
k_diss_metal_ref  0.79   1.32   1.89  1.89
D_ref             1.21   1.20   1.43  1.43
metal_thickness   1.26   1.43   1.40  1.43
trap_vacancy_N_T  1.07   1.42   0.98  1.42
K_s_ref           0.58   1.04   1.31  1.31
trap_vacancy_E_b  1.31   0.77   0.69  1.31
H_s               1.19   0.62   0.74  1.19
trap_carbide_E_b  1.07   0.76   0.68  1.07
oxide_thickness   1.06   0.85   0.94  1.06
H_eq              1.00   0.89   0.58  1.00

[oxide]  floor: 773K=0.094  1073K=0.091  1273K=0.100
                        773K  1073K  1273K  best
parameter                                       
H_sol_ox                2.67   5.48   5.42  5.48
oxide_t

In [13]:
# theta is the metric temperature does NOT dominate — worth its own look
piv_t = (tidy[tidy.metric == 'theta']
         .pivot_table(index=['regime', 'parameter'], columns='temperature_K',
                      values='delta_over_floor'))
piv_t.columns = [f"{c:.0f}K" for c in piv_t.columns]
piv_t['best'] = piv_t.max(axis=1)
print("delta_over_floor for theta (surface coverage) — top 8 per regime\n")
for reg in REGIME_LABELS:
    print(f"[{reg}]")
    print(piv_t.loc[reg].sort_values('best', ascending=False).round(2).head(8).to_string())
    print()

delta_over_floor for theta (surface coverage) — top 8 per regime

[surface]
                  773K  1073K  1273K  best
parameter                                 
H_eq              2.22   0.84   0.81  2.22
H_sol_ox          0.91   2.07   1.98  2.07
P_upstream        1.74   1.53   1.29  1.74
K_eq_ref          1.35   1.33   0.71  1.35
trap_vacancy_N_T  0.78   1.29   0.66  1.29
oxide_thickness   1.02   1.21   0.79  1.21
trap_carbide_E_b  1.13   0.83   0.71  1.13
lattice_density   0.87   1.07   0.66  1.07

[oxide]
                      773K  1073K  1273K  best
parameter                                     
P_upstream            1.55   2.11   1.85  2.11
K_eq_ref              1.17   1.97   1.74  1.97
H_eq                  1.62   1.22   0.99  1.62
trap_dislocation_E_b  1.00   0.59   1.23  1.23
D_ox_ref              0.81   1.22   0.86  1.22
f_pinhole             0.86   1.15   0.83  1.15
trap_carbide_E_b      0.70   1.14   0.81  1.14
trap_gb_N_T           0.65   1.14   1.03  1.14

[metal]
      

In [14]:
# Manifest of everything saved
import glob
print("Saved CSV files under", os.path.abspath(RESULTS_DIR), ":")
for f in sorted(glob.glob(os.path.join(RESULTS_DIR, "**", "*.csv"), recursive=True)):
    print("  ", os.path.relpath(f, RESULTS_DIR), f"({os.path.getsize(f)//1024} KB)")

Saved CSV files under incoloy802_cr2o3/sa_results :
   compare_delta_flux_T1073.csv (0 KB)
   compare_delta_flux_T1273.csv (0 KB)
   compare_delta_flux_T773.csv (0 KB)
   compare_delta_permeability_T1073.csv (0 KB)
   compare_delta_permeability_T1273.csv (0 KB)
   compare_delta_permeability_T773.csv (0 KB)
   compare_delta_theta_T1073.csv (0 KB)
   compare_delta_theta_T1273.csv (0 KB)
   compare_delta_theta_T773.csv (0 KB)
   compare_pawn_median_flux_T1073.csv (0 KB)
   compare_pawn_median_flux_T1273.csv (0 KB)
   compare_pawn_median_flux_T773.csv (0 KB)
   compare_pawn_median_permeability_T1073.csv (0 KB)
   compare_pawn_median_permeability_T1273.csv (0 KB)
   compare_pawn_median_permeability_T773.csv (0 KB)
   compare_pawn_median_theta_T1073.csv (0 KB)
   compare_pawn_median_theta_T1273.csv (0 KB)
   compare_pawn_median_theta_T773.csv (0 KB)
   delta_over_floor_by_T.csv (9 KB)
   master_clusters.csv (15133 KB)
   routeB_givendata.csv (127 KB)
   scans/T1073/scan_metal.csv (1964 KB)
 